# 02 — Merge parish-level accessibility results

## Purpose

Merge the seven final parish-level CSV files generated by Notebook 01 into a single
building-level dataset, perform quality-control checks and export the consolidated file
used by the final analysis notebook.

## Inputs

Seven final parish-level CSV files generated by Notebook 01:

- `aldoar.csv`
- `bonfim.csv`
- `campanha.csv`
- `cedofeita.csv`
- `lordelo.csv`
- `paranhos.csv`
- `ramalde.csv`

## Output

- `building_accessibility_all_parishes.csv`

## Scope

This notebook performs only:

1. file validation;
2. concatenation;
3. geometry reconstruction and validation;
4. numeric type conversion;
5. duplicate checks;
6. calculation of service-validity and functional-diversity fields;
7. export of the consolidated dataset.

Exploratory maps, widgets, filtered alternatives and parish summary tables are not included.

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely import wkt

In [ ]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve().parent
PARISH_RESULTS_DIR = PROJECT_ROOT / "data" / "outputs"
CONSOLIDATED_OUTPUT = (
    PARISH_RESULTS_DIR / "building_accessibility_all_parishes.csv"
)

METRIC_CRS = "EPSG:3763"

PARISH_FILES = {
    "Aldoar, Foz do Douro e Nevogilde": "aldoar.csv",
    "Bonfim": "bonfim.csv",
    "Campanhã": "campanha.csv",
    "Cedofeita, Santo Ildefonso, Sé, Miragaia, São Nicolau e Vitória": "cedofeita.csv",
    "Lordelo do Ouro e Massarelos": "lordelo.csv",
    "Paranhos": "paranhos.csv",
    "Ramalde": "ramalde.csv",
}

SERVICE_CATEGORY_COLUMNS = [
    "Centro Saude",
    "Farmacias",
    "Hospitais",
    "Supermercados",
    "Bancos",
    "CTT",
    "Parques e jardins",
]

NUMERIC_COLUMNS = [
    "area",
    "dist_to_network",
    "component_id",
    "distancia_media_servicos",
    "distancia_minima_servico",
    "tempo_medio_servicos_seg",
    "tempo_minimo_servico_seg",
    "numero_servicos_proximos",
    *SERVICE_CATEGORY_COLUMNS,
]

In [ ]:
# ---------------------------------------------------------------------
# Validate the seven parish files
# ---------------------------------------------------------------------

expected_paths = {
    parish_name: PARISH_RESULTS_DIR / filename
    for parish_name, filename in PARISH_FILES.items()
}

missing_files = [
    str(path)
    for path in expected_paths.values()
    if not path.exists()
]

if missing_files:
    missing_text = "\n".join(missing_files)
    raise FileNotFoundError(
        "The following parish result files are missing:\n"
        f"{missing_text}"
    )

if len(expected_paths) != 7:
    raise ValueError("Exactly seven parish-level files are required.")

print("All seven parish-level result files were found.")

In [ ]:
# ---------------------------------------------------------------------
# Read and merge the parish-level results
# ---------------------------------------------------------------------

parish_frames = []

for parish_name, file_path in expected_paths.items():
    parish_frame = pd.read_csv(file_path, low_memory=False)
    parish_frame["parish_name"] = parish_name
    parish_frame["source_file"] = file_path.name
    parish_frames.append(parish_frame)

merged = pd.concat(parish_frames, ignore_index=True)

print(f"Parish files merged: {len(parish_frames)}")
print(f"Rows before quality control: {len(merged):,}")

In [ ]:
# ---------------------------------------------------------------------
# Validate the minimum required schema
# ---------------------------------------------------------------------

required_columns = {
    "osm_id",
    "geometry_wkt",
    "numero_servicos_proximos",
    "tempo_medio_servicos_seg",
    *SERVICE_CATEGORY_COLUMNS,
}

missing_columns = sorted(required_columns.difference(merged.columns))

if missing_columns:
    raise KeyError(
        "The merged parish files are missing required columns: "
        + ", ".join(missing_columns)
    )

In [ ]:
# ---------------------------------------------------------------------
# Reconstruct and validate geometry
# ---------------------------------------------------------------------

def parse_wkt_geometry(value):
    if pd.isna(value) or not str(value).strip():
        return None
    try:
        return wkt.loads(value)
    except Exception:
        return None

merged["geometry"] = merged["geometry_wkt"].apply(parse_wkt_geometry)

geodata = gpd.GeoDataFrame(
    merged,
    geometry="geometry",
    crs=METRIC_CRS,
)

missing_geometry_count = int(geodata.geometry.isna().sum())
geodata = geodata.dropna(subset=["geometry"]).copy()

invalid_geometry_mask = ~geodata.geometry.is_valid
invalid_before_repair = int(invalid_geometry_mask.sum())

if invalid_before_repair > 0:
    geodata.loc[invalid_geometry_mask, "geometry"] = (
        geodata.loc[invalid_geometry_mask, "geometry"].buffer(0)
    )

geodata = geodata[
    geodata.geometry.notna()
    & ~geodata.geometry.is_empty
    & geodata.geometry.is_valid
].copy()

print(f"Missing geometries removed: {missing_geometry_count:,}")
print(f"Invalid geometries before repair: {invalid_before_repair:,}")
print(f"Valid geometries retained: {len(geodata):,}")

In [ ]:
# ---------------------------------------------------------------------
# Convert analytical fields to numeric values
# ---------------------------------------------------------------------

for column in NUMERIC_COLUMNS:
    if column in geodata.columns:
        geodata[column] = pd.to_numeric(
            geodata[column],
            errors="coerce",
        )

In [ ]:
# ---------------------------------------------------------------------
# Derive final quality-control fields
# ---------------------------------------------------------------------

geodata["sem_servico_valido"] = (
    geodata["numero_servicos_proximos"].fillna(0).eq(0)
    | geodata["tempo_medio_servicos_seg"].isna()
)

geodata["diversidade_servicos"] = (
    geodata[SERVICE_CATEGORY_COLUMNS]
    .fillna(0)
    .gt(0)
    .sum(axis=1)
)

In [ ]:
# ---------------------------------------------------------------------
# Duplicate checks
# ---------------------------------------------------------------------

duplicate_rows = int(
    pd.DataFrame(geodata.drop(columns="geometry")).duplicated().sum()
)
duplicate_osm_ids = int(geodata["osm_id"].duplicated().sum())

if duplicate_osm_ids > 0:
    duplicated_ids = (
        geodata.loc[
            geodata["osm_id"].duplicated(keep=False),
            ["osm_id", "parish_name", "source_file"],
        ]
        .sort_values("osm_id")
    )
    raise ValueError(
        f"{duplicate_osm_ids} duplicated osm_id values were found. "
        "Inspect the duplicated building identifiers before export."
    )

print(f"Duplicated complete rows: {duplicate_rows:,}")
print(f"Duplicated osm_id values: {duplicate_osm_ids:,}")

In [ ]:
# ---------------------------------------------------------------------
# Quality-control summary
# ---------------------------------------------------------------------

parish_counts = (
    geodata.groupby(["parish_name", "source_file"])
    .size()
    .reset_index(name="building_count")
    .sort_values("parish_name")
)

total_buildings = len(geodata)
buildings_without_service = int(
    geodata["sem_servico_valido"].sum()
)
buildings_with_service = (
    total_buildings - buildings_without_service
)

print(f"Total buildings: {total_buildings:,}")
print(f"Buildings with an accessible service: {buildings_with_service:,}")
print(f"Buildings without an accessible service: {buildings_without_service:,}")
print()
print("Buildings by parish:")
display(parish_counts)

In [ ]:
# ---------------------------------------------------------------------
# Reproducibility checks based on the final study dataset
# ---------------------------------------------------------------------
# These checks correspond to the final consolidated dataset reported
# in the research notebook. Update them only if the final manuscript
# dataset is intentionally revised.

EXPECTED_TOTAL_BUILDINGS = 31_337
EXPECTED_BUILDINGS_WITHOUT_SERVICE = 569

assert total_buildings == EXPECTED_TOTAL_BUILDINGS, (
    f"Expected {EXPECTED_TOTAL_BUILDINGS:,} buildings, "
    f"but found {total_buildings:,}."
)

assert (
    buildings_without_service
    == EXPECTED_BUILDINGS_WITHOUT_SERVICE
), (
    f"Expected {EXPECTED_BUILDINGS_WITHOUT_SERVICE:,} buildings "
    f"without an accessible service, but found "
    f"{buildings_without_service:,}."
)

assert geodata["diversidade_servicos"].between(0, 7).all()
assert geodata["osm_id"].notna().all()
assert geodata.geometry.is_valid.all()

print("All reproducibility checks passed.")

In [ ]:
# ---------------------------------------------------------------------
# Export the consolidated dataset
# ---------------------------------------------------------------------

geodata["geometry_wkt"] = geodata.geometry.to_wkt()

export = pd.DataFrame(
    geodata.drop(columns="geometry")
)

CONSOLIDATED_OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True,
)

export.to_csv(
    CONSOLIDATED_OUTPUT,
    index=False,
    encoding="utf-8",
)

print(f"Consolidated output written to: {CONSOLIDATED_OUTPUT}")
print(f"Rows exported: {len(export):,}")

## Final output

The consolidated file contains all 31,337 buildings from the seven parish-level
runs and is the input for Notebook 03.

The seven parish CSV files remain outputs of Notebook 01 and inputs of this notebook.
They are not raw source data and do not need separate source-data documentation.